# Run AI Studio in Google Colab

This notebook runs your Next.js app in Colab and exposes it with a public URL.

Notes:
- URL is temporary (session-based).
- Best for demo/testing.
- Keep mock APIs for lowest GPU usage.

In [ ]:
REPO_URL = "https://github.com/your-username/your-repo.git"  # change this
REPO_DIR = "/content/web_ai"
NODE_MAJOR = "20"

## 1) Install Node.js

In [ ]:
!apt-get update -y
!apt-get install -y curl git
!curl -fsSL https://deb.nodesource.com/setup_${NODE_MAJOR}.x | bash -
!apt-get install -y nodejs
!node -v
!npm -v

## 2) Clone project

In [ ]:
import os, shutil
if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)
!git clone "$REPO_URL" "$REPO_DIR"
%cd /content/web_ai
!ls

## 3) Install dependencies

In [ ]:
%cd /content/web_ai
!npm install

## 4) (Optional) Add .env.local for real model endpoints

Skip this in mock mode.

In [ ]:
%%bash
cd /content/web_ai
cat > .env.local << 'EOF'
# Example only. Keep empty for mock mode.
# CHAT_ENDPOINT=https://your-chat-endpoint
# CHAT_API_KEY=your_key
# IMAGE_ENDPOINT=https://your-image-endpoint
# VIDEO_ENDPOINT=https://your-video-endpoint
# TRANSITION_ENDPOINT=https://your-transition-endpoint
EOF
cat .env.local

## 5) Start Next.js app

In [ ]:
import os, subprocess
os.chdir('/content/web_ai')
log = open('/content/next.log', 'w')
proc = subprocess.Popen(
    ['npm', 'run', 'dev', '--', '--hostname', '0.0.0.0', '--port', '3000'],
    stdout=log,
    stderr=subprocess.STDOUT
)
print('Next.js PID:', proc.pid)
print('Log file: /content/next.log')

## 6) Expose public URL (Cloudflare Tunnel)

Run this cell and open the generated `https://...trycloudflare.com` URL.

In [ ]:
!wget -q -O /content/cloudflared.deb https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i /content/cloudflared.deb
!cloudflared tunnel --url http://localhost:3000 --no-autoupdate

## 7) Debug logs (optional)

In [ ]:
!tail -n 120 /content/next.log